In [5]:
from  ..middleWare import *
#初始化模型
load_dotenv(override=True)

DEEPSEEK_API_KEY=os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL=os.getenv('DEEPSEEK_BASE_URL')
model=init_chat_model(
    model='deepseek-v4-flash',
    model_provider='deepseek',
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={'thinking':{"type":'disabled'}},
)

In [3]:
#使用PIIMiddleware对发送的隐私信息进行加密，内置了5种需要加密的信息,邮件，银行卡号，url地址，设备mac地址，ip地址
#加密策略内置了四种，redact用字符串{REDACTED_TYPE}替换信息,mask用*号掩盖部分信息，hash用哈希值替换信息，block直接抛出异常
myagent=create_agent(
    model=model,
    middleware=[
        PIIMiddleware('email',strategy='redact',apply_to_input=True),
        PIIMiddleware('credit_card',strategy='mask',apply_to_input=True),
        PIIMiddleware('url',strategy='hash',apply_to_input=True),
        PIIMiddleware('mac_address',strategy='mask',apply_to_input=True),
        PIIMiddleware('ip',strategy='block',apply_to_input=True),
    ]
)
messages = [HumanMessage('''
              帮我向 156168188@qq.com 发送一封邮件
            同时查看银行卡号： 5105-1051-0510-5100 的余额
            访问 https://localhost:12345
            确认这是不是 MAC地址： 11-11-11-11-11-11  ''')]
response=myagent.invoke({
    'messages':messages
})
for msg in response['messages']:
    msg.pretty_print()
try:
    response1 = myagent.invoke({
    "messages": [HumanMessage("看看这个 IP 能不能 ping 通：192.168.10.1")]
        })
except Exception as e:
    print('=' * 30, '-> 抛异常 <-', '=' * 30)
    print(f"检测到IP，抛出异常：{e}")

================================ Human Message =================================


              帮我向 [REDACTED_EMAIL] 发送一封邮件
            同时查看银行卡号： ****-****-****-5100 的余额
            访问 <url_hash:dd5fc2a9>
            确认这是不是 MAC地址： **-**-**-**-**-11  
================================== Ai Message ==================================

我无法执行以下操作，因为它们涉及个人信息访问和实际操作：

1. **发送邮件**：我无法访问外部电子邮件系统或代表你发送邮件。你需要使用自己的电子邮件客户端（如 Gmail、Outlook）手动发送。
2. **查询银行卡余额**：我无法访问银行系统、查看你的账户信息或处理金融数据。请通过银行官方应用、网站或客服查询。
3. **访问 URL**：我无法直接访问或打开链接（包括 `<url_hash:dd5fc2a9>`）。这是出于安全原因，我无法执行外部网络请求。
4. **确认 MAC 地址**：MAC 地址通常格式为 6 组两位十六进制数（如 `00-1A-2B-3C-4D-5E`）。你提供的格式 `**-**-**-**-**-11` 中带有星号（`*`），不是有效 MAC 地址。有效的 MAC 地址应只包含十六进制字符（0-9, A-F）和分隔符。请提供完整的地址。

如果你需要其他协助（例如解释如何自己完成这些操作），请告诉我！
============================== -> 抛异常 <- ==============================
检测到IP，抛出异常：Detected 1 instance(s) of ip in text content


In [8]:
#定义手机号检测函数
def detect_phone_number(number:str):
    return [
        {
            'text':m.group(),
            'start':m.start(),
            'end': m.end()
        } for m in re.finditer(r'[0-9]{11}',number)
    ]
content='hyz的手机号是12345678910'
result=detect_phone_number(content)
print(result)

[{'text': '12345678910', 'start': 8, 'end': 19}]


In [13]:
#中间件使用自定义检测器检测手机号
myagent2=create_agent(
    model=model,
    middleware=[
        PIIMiddleware('phone-number',strategy='redact',apply_to_input=True,detector=detect_phone_number),
    ]
)
response=myagent2.invoke({
    'messages':HumanMessage('''帮我提取下面的手机号：
    13890232207''')
})
for msg in response['messages']:
    msg.pretty_print()

================================ Human Message =================================

帮我提取下面的手机号：
    [REDACTED_PHONE-NUMBER]
================================== Ai Message ==================================

我无法提取你提到的手机号，因为内容中显示为 `[REDACTED_PHONE-NUMBER]`，这表示该手机号已被脱敏或隐藏。如果你有包含实际手机号的文本，请提供具体内容，我可以帮你识别和提取其中的手机号。
